In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[0:5]:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_074_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_070_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_035_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_042_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/label/20240529_EO4_RES2_fl_pid_012_label.tif
/kaggle/input/competitions/aisehack-theme-1/data/split/pred.txt
/kaggle/input/competitions/aisehack-theme-1/data/split/val.txt
/kaggle/input/competitions/aisehack-theme-1/data/split/test.txt
/kaggle/input/competitions/aisehack-theme-1/data/split/train.txt
/kaggle/input/competitions/aisehack-theme-1/data/prediction/image/20240529_EO4_RES2_fl_pid_089_image.tif
/kaggle/input/competitions/aisehack-theme-1/data/prediction/image/20240529_EO4_RES2_fl_pid_095_image.tif
/kaggle/input/competitions/aisehack-theme-1/data/prediction/imag

# Knowing The Data Directory

In [2]:
train = []
val = []
test = []
pred = []

data_root = '/kaggle/input/competitions/aisehack-theme-1/data/split'

def data(name: str, l: list) -> list:
    with open(f'{data_root}/{name}.txt', 'r') as file:
        for line in file.readlines():
            l.append((lambda x: int(x))(line[-3:]))
    return 

data('train', train)
data('val', val)
data('test', test)
data('pred', pred)

print(f"Train IDs: {train}")
print(f"Validation IDs: {val}")
print(f"Test IDs: {test}")
print(f"Prediciton IDs: {pred}")

# The above info is from data_root/split

# data_root/image has train, val, test images
# data_root/label has train, val, test labels
# data_root/prediction/image has prediction images without labels for which labels are to be generated

Train IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
Validation IDs: [60, 61, 62, 63, 64, 65, 66, 67, 68, 69]
Test IDs: [70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
Prediciton IDs: [80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98]


# Install Dependencies

In [3]:
!pip install -q numpy pandas lightning segmentation-models-pytorch rasterio kagglehub torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 9.9 MB/s eta 0:00:00


# Imports & Configuration

In [4]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from torchvision.transforms import v2
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
import segmentation_models_pytorch as smp
import kagglehub

# --- Configuration ---
DATA_ROOT = '/kaggle/input/competitions/aisehack-theme-1/data'
OUT_DIR = '/kaggle/working/predictions'
MODEL_SAVE_DIR = '/kaggle/working/model_export'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

EPOCHS = 10
BATCH_SIZE = 4
LR = 1e-4

# Normalization constants (SAR + Optical bands)
MEANS = np.array([841.1257, 371.6175, 1734.1789, 1588.3142, 1742.8452, 1218.5551])
STDS = np.array([473.7090, 170.3611, 623.0474, 612.8465, 564.5835, 528.0894])

# Dataset Definition

In [5]:
class FloodDataset(Dataset):
    def __init__(self, split_name, data_root, transform=None, is_test=False):
        self.data_root = Path(data_root)
        self.transform = transform
        self.is_test = is_test
        
        with open(self.data_root / f'split/{split_name}.txt', 'r') as f:
            self.file_ids = [line.strip() for line in f.readlines()]
            
        if self.is_test:
            self.image_dir = self.data_root / 'prediction/image'
            self.label_dir = None
        else:
            self.image_dir = self.data_root / 'image'
            self.label_dir = self.data_root / 'label'

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        file_id = self.file_ids[idx]
        
        # Load 6-Channel Image
        img_pattern = f"*{file_id}_image.tif"
        img_path = list(self.image_dir.glob(img_pattern))[0]
        with rasterio.open(img_path) as src:
            image = src.read().astype(np.float32) # [6, H, W]
            
        # Z-Score Normalization
        image = (image - MEANS[:, None, None]) / STDS[:, None, None]
        image_tensor = torch.from_numpy(image).float()
        
        if self.is_test:
            if self.transform:
                image_tensor = self.transform(image_tensor)
            return image_tensor, str(img_path)
            
        # Load Label Mask
        lbl_pattern = f"*{file_id}_label.tif"
        lbl_path = list(self.label_dir.glob(lbl_pattern))[0]
        with rasterio.open(lbl_path) as src:
            mask = src.read(1).astype(np.int64) # [H, W]
            
        mask_tensor = torch.from_numpy(mask)
        
        # Wrap mask for Torchvision v2 API to ensure aligned augmentations
        if self.transform:
            mask_tv = tv_tensors.Mask(mask_tensor)
            image_tensor, mask_tensor = self.transform(image_tensor, mask_tv)
            
        return image_tensor, mask_tensor.long()

# Datamodule

In [6]:
class FloodDataModule(pl.LightningDataModule):
    def __init__(self, data_root, batch_size=8):
        super().__init__()
        self.data_root = data_root
        self.batch_size = batch_size
        
        # Modern Torchvision V2 Augmentations
        self.train_transform = v2.Compose([
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.5),
        ])
        self.val_transform = None

    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            self.train_ds = FloodDataset('train', self.data_root, transform=self.train_transform)
            self.val_ds = FloodDataset('val', self.data_root, transform=self.val_transform)
        if stage == 'predict' or stage is None:
            self.pred_ds = FloodDataset('pred', self.data_root, transform=self.val_transform, is_test=True)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=2)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers=2)

    def predict_dataloader(self):
        return DataLoader(self.pred_ds, batch_size=1, shuffle=False, num_workers=2)

In [7]:
class FloodDataModuleV2(pl.LightningDataModule):
    def __init__(self, data_root, batch_size=8):
        super().__init__()
        self.data_root = data_root
        self.batch_size = batch_size

        from torchvision.transforms import v2
        from torchvision import tv_tensors

        self.train_transform = v2.Compose([
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.5),
            v2.RandomApply([v2.RandomRotation(degrees=90)], p=0.5),
            # Simulate sensor noise - safe for 6-channel
            v2.RandomApply([v2.GaussianNoise(mean=0.0, sigma=0.05)], p=0.3),
        ])
        self.val_transform = None

    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            self.train_ds = FloodDataset('train', self.data_root, transform=self.train_transform)
            self.val_ds = FloodDataset('val', self.data_root, transform=self.val_transform)
        if stage == 'predict' or stage is None:
            self.pred_ds = FloodDataset('pred', self.data_root, transform=self.val_transform, is_test=True)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True,
                          num_workers=2, pin_memory=True, persistent_workers=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)

    def predict_dataloader(self):
        return DataLoader(self.pred_ds, batch_size=1, shuffle=False, num_workers=2)

# Addition of Better Loss

In [8]:
class DiceLoss(nn.Module):
    def __init__(self, ignore_index=-1, smooth=1.0):
        super().__init__()
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, logits, targets):
        # Mask out ignored pixels
        valid = targets != self.ignore_index
        targets = targets.clone()
        targets[~valid] = 0

        probs = F.softmax(logits, dim=1)  # [B, C, H, W]
        flood_prob = probs[:, 1]           # probability of flood class

        target_f = targets.float()
        valid_f = valid.float()

        intersection = (flood_prob * target_f * valid_f).sum(dim=(1, 2))
        union = ((flood_prob + target_f) * valid_f).sum(dim=(1, 2))

        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, ignore_index=-1):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, ignore_index=self.ignore_index, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        valid = targets != self.ignore_index
        return focal[valid].mean()


class CombinedLoss(nn.Module):
    def __init__(self, ignore_index=-1, dice_weight=0.5, focal_weight=0.5):
        super().__init__()
        self.dice = DiceLoss(ignore_index=ignore_index)
        self.focal = FocalLoss(ignore_index=ignore_index)
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight

    def forward(self, logits, targets):
        return self.dice_weight * self.dice(logits, targets) + \
               self.focal_weight * self.focal(logits, targets)

# Model

In [9]:
# class FloodSegmentationModel(pl.LightningModule):
#     def __init__(self, learning_rate=1e-4):
#         super().__init__()
#         self.save_hyperparameters()
#         self.learning_rate = learning_rate
        
#         # Define UNet with EfficientNet-b5 (or resnet101)
#         self.model = smp.Unet(
#             encoder_name="resnet101",
#             encoder_weights="imagenet", 
#             in_channels=6, # Handling SAR + Optical
#             classes=2,     # Flood vs No-Flood
#         )
        
#         # Ignore -1 values (unclassified pixels) during loss calculation
#         self.criterion = nn.CrossEntropyLoss(ignore_index=-1)

#     def forward(self, x):
#         return self.model(x)

#     def training_step(self, batch, batch_idx):
#         images, masks = batch
#         logits = self(images)
#         loss = self.criterion(logits, masks)
#         self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
#         return loss

#     def validation_step(self, batch, batch_idx):
#         images, masks = batch
#         logits = self(images)
#         loss = self.criterion(logits, masks)
#         self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
#         return loss
        
#     def predict_step(self, batch, batch_idx):
#         images, paths = batch
#         logits = self(images)
#         # Squeeze down to the specific classes per pixel
#         preds = torch.argmax(logits, dim=1) 
#         return preds, paths

#     def configure_optimizers(self):
#         return torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)

In [10]:
class FloodSegmentationModel(pl.LightningModule):
    def __init__(self, 
                 encoder_name="mit_b3",   # SegFormer backbone - better than resnet101
                 learning_rate=1e-4,
                 class_weights=None):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate

        self.model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights="imagenet",
            in_channels=6,
            classes=2,
        )

        self.criterion = CombinedLoss(ignore_index=-1, dice_weight=0.5, focal_weight=0.5)

        # Track IoU (what the leaderboard likely uses)
        self.train_tp = torch.zeros(2)
        self.train_fp = torch.zeros(2)
        self.train_fn = torch.zeros(2)
        self.val_tp = torch.zeros(2)
        self.val_fp = torch.zeros(2)
        self.val_fn = torch.zeros(2)

    def forward(self, x):
        return self.model(x)

    def _compute_iou_stats(self, logits, masks):
        preds = torch.argmax(logits, dim=1)
        valid = masks != -1
        tp = torch.zeros(2, device=self.device)
        fp = torch.zeros(2, device=self.device)
        fn = torch.zeros(2, device=self.device)
        for c in range(2):
            pred_c = (preds == c) & valid
            true_c = (masks == c) & valid
            tp[c] = (pred_c & true_c).sum()
            fp[c] = (pred_c & ~true_c).sum()
            fn[c] = (~pred_c & true_c).sum()
        return tp, fp, fn

    def training_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)
        loss = self.criterion(logits, masks)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)
        loss = self.criterion(logits, masks)
        tp, fp, fn = self._compute_iou_stats(logits, masks)
        self.val_tp += tp.cpu()
        self.val_fp += fp.cpu()
        self.val_fn += fn.cpu()
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        iou = self.val_tp / (self.val_tp + self.val_fp + self.val_fn + 1e-6)
        miou = iou.mean()
        flood_iou = iou[1]  # IoU for flood class specifically
        self.log('val_mIoU', miou, prog_bar=True)
        self.log('val_flood_IoU', flood_iou, prog_bar=True)
        self.val_tp = torch.zeros(2)
        self.val_fp = torch.zeros(2)
        self.val_fn = torch.zeros(2)

    def predict_step(self, batch, batch_idx):
        images, paths = batch
        logits = self(images)
        preds = torch.argmax(logits, dim=1)
        return preds, paths

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS, eta_min=1e-6
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}

# Training

## ResNet101/Efficient-B5

In [11]:
# pl.seed_everything(42)

# datamodule = FloodDataModule(data_root=DATA_ROOT, batch_size=BATCH_SIZE)
# model = FloodSegmentationModel(learning_rate=LR)

# # Callback to automatically save the weights that performed best on validation
# checkpoint_callback = ModelCheckpoint(
#     dirpath=MODEL_SAVE_DIR,
#     filename='best-flood-model',
#     save_top_k=1,
#     monitor='val_loss',
#     mode='min'
# )

# # Trainer abstraction 
# trainer = pl.Trainer(
#     max_epochs=EPOCHS,
#     accelerator='auto',
#     devices=1,
#     precision="16-mixed",
#     accumulate_grad_batches=2,
#     callbacks=[checkpoint_callback],
# )

# print("Starting Training")
# trainer.fit(model, datamodule=datamodule)

## 

## SENet+ResNext/MIT-B3

In [12]:
!pip install timm

In [13]:
# Cell: Try these encoder options one at a time, pick the one with best val_mIoU

ENCODER_OPTIONS = [
    "mit_b3",          # SegFormer - excellent for segmentation, needs: pip install timm
    "mit_b5",
    "efficientnet-b5", # Good balance of speed/accuracy
    "resnet101",       # Your current baseline
    "se_resnext101_32x4d",  # SE-Net with ResNext, often beats resnet101
]

ENCODER_TO_TRY = ENCODER_OPTIONS[1]

pl.seed_everything(42)

datamodule_v2 = FloodDataModuleV2(data_root=DATA_ROOT, batch_size=BATCH_SIZE)
model_v2 = FloodSegmentationModel(
    encoder_name=ENCODER_TO_TRY,
    learning_rate=LR,
)

checkpoint_callback_v2 = ModelCheckpoint(
    dirpath=MODEL_SAVE_DIR,
    filename=f'best-{ENCODER_TO_TRY}',
    save_top_k=1,
    monitor='val_mIoU',   # NOW monitoring IoU not loss
    mode='max'            # higher is better
)

trainer_v2 = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator='auto',
    devices=1,
    precision="16-mixed",
    accumulate_grad_batches=2,
    callbacks=[checkpoint_callback_v2],
)

trainer_v2.fit(model_v2, datamodule=datamodule_v2)

Seed set to 42


config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-15 21:06:25.465100: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773608785.906896      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773608786.022518      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773608787.076143      24 computation_placer.cc:177] computation placer

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ Unet         │ 84.7 M │ train │     0 │
│ 1 │ criterion │ CombinedLoss │      0 │ train │     0 │
└───┴───────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 84.7 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 84.7 M                                                                                               
Total estimated model params size (MB): 338                                                                        
Modules in train mode: 1086                                                                                        
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches 
(15) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if
you want to see logs for the training epoch.

`Trainer.fit` stopped: `max_epochs=10` reached.


# Submission

In [14]:
def mask_to_rle(mask):
    """Convert binary mask to Kaggle RLE format."""
    pixels = mask.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

# print("Starting Inference on Prediction Data...")
# # trainer.predict auto-loads the best checkpoint generated above
# predictions = trainer.predict(model, datamodule=datamodule, ckpt_path='best')

# submission_rows = []

# # Unpack batches generated by `predict_step`
# for preds_batch, paths_batch in predictions:
#     preds_numpy = preds_batch.cpu().numpy() # Shape: [B, H, W]
    
#     for i in range(len(paths_batch)):
#         file_path = paths_batch[i]
#         img_id = Path(file_path).name.replace("_image.tif", "")
        
#         binary_mask = (preds_numpy[i] == 1).astype(np.uint8)
#         rle_string = mask_to_rle(binary_mask)
        
#         submission_rows.append({
#             "id": img_id,
#             "rle_mask": rle_string
#         })

# df_sub = pd.DataFrame(submission_rows)
# # Kaggle constraint: empty masks must be registered as "0"
# df_sub = df_sub.replace("", 0).fillna(0)
# df_sub.to_csv(OUT_DIR + "/submission.csv", index=False)
# print(f"✅ RLE Submission saved to: {OUT_DIR}/submission.csv")

## Test Time Augmentation @ Inference

In [15]:
def predict_with_tta(model, dataloader, device):
    """
    Run inference with TTA: original + hflip + vflip + hflip+vflip.
    Averages softmax probabilities before taking argmax.
    """
    model.eval()
    all_preds = []
    all_paths = []

    with torch.no_grad():
        for images, paths in dataloader:
            images = images.to(device)
            B = images.shape[0]

            # 4 augmentation versions
            aug_logits = []

            # Original
            aug_logits.append(torch.softmax(model(images), dim=1))
            # Horizontal flip
            aug_logits.append(torch.softmax(model(torch.flip(images, dims=[3])), dim=1).flip(dims=[3]))
            # Vertical flip
            aug_logits.append(torch.softmax(model(torch.flip(images, dims=[2])), dim=1).flip(dims=[2]))
            # Both flips
            aug_logits.append(torch.softmax(model(torch.flip(images, dims=[2, 3])), dim=1).flip(dims=[2, 3]))

            # Average and argmax
            avg_probs = torch.stack(aug_logits).mean(dim=0)
            preds = torch.argmax(avg_probs, dim=1)

            all_preds.append(preds.cpu())
            all_paths.extend(paths)

    return all_preds, all_paths


# Use it instead of trainer.predict:
best_model_path = os.path.join(MODEL_SAVE_DIR, f"best-{ENCODER_TO_TRY}.ckpt")
loaded_model = FloodSegmentationModel.load_from_checkpoint(best_model_path)
loaded_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded_model = loaded_model.to(device)

datamodule_v2.setup('predict')
pred_loader = datamodule_v2.predict_dataloader()

tta_preds, tta_paths = predict_with_tta(loaded_model, pred_loader, device)

In [16]:
submission_rows = []

for preds_tensor, file_path in zip(tta_preds, tta_paths):
    preds_numpy = preds_tensor.squeeze(0).numpy()  # [H, W]
    img_id = Path(file_path).name.replace("_image.tif", "")
    binary_mask = (preds_numpy == 1).astype(np.uint8)
    rle_string = mask_to_rle(binary_mask)
    submission_rows.append({"id": img_id, "rle_mask": rle_string})

df_sub = pd.DataFrame(submission_rows)
df_sub = df_sub.replace("", 0).fillna(0)
df_sub.to_csv(OUT_DIR + "/submission_tta.csv", index=False)
print(f"✅ TTA Submission saved: {OUT_DIR}/submission_tta.csv")

✅ TTA Submission saved: /kaggle/working/predictions/submission_tta.csv


# Kaggle Hub Upload (For Inference If Needed)

In [17]:
# First, extract and save the clean PyTorch model weights
best_model_path = os.path.join(MODEL_SAVE_DIR, f"best-{ENCODER_TO_TRY}.ckpt")
loaded_model = FloodSegmentationModel.load_from_checkpoint(best_model_path)

# Construct your specific Kaggle handle using the environment variable
# Handle format: <KAGGLE_USERNAME>/<MODEL_SLUG>/<FRAMEWORK>/<VARIATION>
USERNAME = "devamwppu2000828"
MODEL_SLUG = f"aise-flood-detection-{ENCODER_TO_TRY}" 
FRAMEWORK = "pytorch"
VARIATION = "v1"

handle = f"{USERNAME}/{MODEL_SLUG}/{FRAMEWORK}/{VARIATION}"

print(f"Attempting to upload to Kaggle Hub as: {handle}")
try:
    # This automatically syncs your local folder to Kaggle Models
    kagglehub.model_upload(
        handle=handle,
        local_model_dir=MODEL_SAVE_DIR, 
        version_notes= f"PyTorch Lightning & {ENCODER_TO_TRY} backbone"
    )
    print("✅ Successfully pushed to Kaggle Hub!")
except Exception as e:
    print(f"❌ Upload Failed: {e}")
    print("Troubleshooting: Make sure to replace YOUR_KAGGLE_USERNAME if the environment variable is unset, and ensure your model slug only contains alphanumeric characters/hyphens.")

Attempting to upload to Kaggle Hub as: devamwppu2000828/aise-flood-detection-mit_b5/pytorch/v1
Uploading Model https://api.kaggle.com/models/devamwppu2000828/aise-flood-detection-mit_b5/pytorch/v1 ...
Model 'aise-flood-detection-mit_b5' does not exist or access is forbidden for user 'devamwppu2000828'. Creating or handling Model...
Model 'aise-flood-detection-mit_b5' Created.
Starting upload for file /kaggle/working/model_export/best-mit_b5.ckpt


Uploading: 100%|██████████| 1.02G/1.02G [00:07<00:00, 130MB/s]

Upload successful: /kaggle/working/model_export/best-mit_b5.ckpt (971MB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/devamwppu2000828/aise-flood-detection-mit_b5/pytorch/v1
✅ Successfully pushed to Kaggle Hub!
